In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

sns.set_style('whitegrid')

In [ ]:
# Load the dataset
try:
    df = pd.read_csv(r"C:\Users\abbas\Downloads\archive (2)\Telco.csv")
except FileNotFoundError:
    df = pd.read_csv('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-ic14/master/WA_Fn-UseC_-Telco-Customer-Churn.csv')

if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

if 'Churn' in df.columns:
    df['Churn'] = df['Churn'].replace({'Yes': 1, 'No': 0})
elif 'churn' in df.columns:
    df['Churn'] = df['Churn'].replace({'Yes': 1, 'No': 0})

for col in ['TotalCharges', 'MonthlyCharges', 'tenure']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

if 'TotalCharges' in df.columns:
    df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


def prepare_frame(df_in):
    df_out = df_in.copy()
    for col in ['TotalCharges', 'MonthlyCharges', 'tenure']:
        if col in df_out.columns:
            df_out[col] = pd.to_numeric(df_out[col], errors='coerce')
    for col in df_out.columns:
        if df_out[col].dtype == 'object':
            df_out[col] = df_out[col].fillna('missing').astype(str)
    return df_out


def add_engineered_features(df_in):
    df_out = prepare_frame(df_in)
    if 'MonthlyCharges' in df_out.columns and 'tenure' in df_out.columns:
        df_out['charge_per_tenure'] = df_out['MonthlyCharges'] / (df_out['tenure'].replace(0, np.nan) + 1)
        df_out['charge_per_tenure'] = df_out['charge_per_tenure'].fillna(df_out['charge_per_tenure'].median())
    service_columns = [col for col in ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'] if col in df_out.columns]
    if service_columns:
        service_frame = df_out[service_columns].replace({'Yes': 1, 'No': 0, 'No internet service': 0, 'No phone service': 0})
        service_frame = service_frame.apply(pd.to_numeric, errors='coerce').fillna(0)
        df_out['num_additional_services'] = service_frame.sum(axis=1)
    return df_out

engineered_X = add_engineered_features(X)
engineered_X_train, engineered_X_test = train_test_split(
    engineered_X, test_size=0.25, random_state=42, stratify=y
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, make_column_selector(dtype_include=['int64', 'float64'])),
        ('cat', categorical_transformer, make_column_selector(dtype_exclude=['int64', 'float64']))
    ]
)

logistic_model = Pipeline([
    ('prepare', FunctionTransformer(add_engineered_features, validate=False)),
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

random_forest = Pipeline([
    ('prepare', FunctionTransformer(add_engineered_features, validate=False)),
    ('preprocess', preprocess),
    ('model', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
])

xgb_model = Pipeline([
    ('prepare', FunctionTransformer(add_engineered_features, validate=False)),
    ('preprocess', preprocess),
    ('model', XGBClassifier(n_estimators=250, learning_rate=0.1, max_depth=4, random_state=42, n_jobs=-1))
])

logistic_model.fit(X_train, y_train)
pred_logistic = logistic_model.predict(X_test)

random_forest.fit(X_train, y_train)
pred_rf = random_forest.predict(X_test)

xgb_model.fit(X_train, y_train)
pred_xgb = xgb_model.predict(X_test)


def summarize(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    print(f'\n{name}')
    print('Accuracy:', round(acc, 4))
    print('Precision:', round(prec, 4))
    print('Recall:', round(rec, 4))
    print('F1:', round(f1, 4))
    print(classification_report(y_true, y_pred, zero_division=0))

summarize('Logistic Regression (baseline)', y_test, pred_logistic)
summarize('Random Forest', y_test, pred_rf)
summarize('XGBoost', y_test, pred_xgb)

# Feature importance comparison
rf_importance = pd.DataFrame({
    'feature': random_forest.named_steps['preprocess'].get_feature_names_out(),
    'importance': random_forest.named_steps['model'].feature_importances_
}).sort_values('importance', ascending=False).head(10)

xgb_importance = pd.DataFrame({
    'feature': xgb_model.named_steps['preprocess'].get_feature_names_out(),
    'importance': xgb_model.named_steps['model'].feature_importances_
}).sort_values('importance', ascending=False).head(10)

print('\nTop features for Random Forest:')
print(rf_importance)
print('\nTop features for XGBoost:')
print(xgb_importance)

# Write a short comparison summary
comparison_summary = """Random Forest builds many decision trees from different bootstrapped samples and averages their predictions, which reduces variance and helps prevent overfitting. XGBoost builds trees sequentially, where each new tree corrects the mistakes of the previous ones, making it more focused on hard cases. In practice, Random Forest is robust and simple, while XGBoost often delivers stronger performance when tuned carefully. Both are ensemble methods, but they combine trees in different ways: Random Forest uses parallel independent trees, while XGBoost uses boosting with iterative refinement."""
print('\nEnsemble explanation:')
print(comparison_summary)

# Save comparison table as markdown
comparison_table = pd.DataFrame([
    {'model': 'Logistic Regression', 'metric': 'F1', 'score': round(f1_score(y_test, pred_logistic, zero_division=0), 4)},
    {'model': 'Random Forest', 'metric': 'F1', 'score': round(f1_score(y_test, pred_rf, zero_division=0), 4)},
    {'model': 'XGBoost', 'metric': 'F1', 'score': round(f1_score(y_test, pred_xgb, zero_division=0), 4)},
])
comparison_table.to_markdown('comparison_table.md', index=False)
print('\nSaved comparison table to comparison_table.md')

model_path = Path('churn_pipeline.joblib')
joblib.dump(logistic_model, model_path)
print(f'\nSaved logistic pipeline to {model_path.resolve()}')

C:\Users\abbas\AppData\Local\Temp\ipykernel_15528\2273094220.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Churn'] = df['Churn'].replace({'Yes': 1, 'No': 0})
C:\Users\abbas\AppData\Local\Temp\ipykernel_15528\2273094220.py:48: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  service_frame = df_out[service_columns].replace({'Yes': 1, 'No': 0, 'No internet service': 0, 'No phone service': 0})
C:\Users\abbas\AppData\Local\Temp\ipykernel_15528\2273094220.py:48: FutureWarning: Downcasting behavior in `replace` is deprecated and 


Pipeline without engineered features
Accuracy: 0.8081
Precision: 0.6641
Recall: 0.5589
F1: 0.607
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1294
           1       0.66      0.56      0.61       467

    accuracy                           0.81      1761
   macro avg       0.76      0.73      0.74      1761
weighted avg       0.80      0.81      0.80      1761


Pipeline with engineered features
Accuracy: 0.8047
Precision: 0.6713
Recall: 0.5161
F1: 0.5835
              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1294
           1       0.67      0.52      0.58       467

    accuracy                           0.80      1761
   macro avg       0.76      0.71      0.73      1761
weighted avg       0.79      0.80      0.80      1761


Manual preprocessing baseline
Accuracy: 0.8058
Precision: 0.6731
Recall: 0.5203
F1: 0.587
              precision    recall  f1-score   support

      